# HZa Tagger — Performance Notebook

Loads scored test H5, computes ROC / AUC, score distributions, and WP efficiencies.

**Prerequisites:** run `analysis/scripts/eval_to_h5.py` to produce the scored H5.

In [ ]:
import sys; sys.path.insert(0, '../..')

import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import roc_curve, auc
import h5py

from common.io import JETS_DATASET, LABELS_DATASET

%matplotlib inline
plt.rcParams['figure.dpi'] = 120

In [ ]:
SCORES_FILE = '../../data/test_scores.h5'  # adjust path

with h5py.File(SCORES_FILE) as f:
    jets   = f[JETS_DATASET][:]
    labels = f[LABELS_DATASET]['a_jet'][:]
    scores = f['scores'][:, 1]  # P(a_jet)

print(f'Total jets: {len(labels)}')
print(f'Signal (a-jet): {labels.sum()} ({100*labels.mean():.1f}%)')

In [ ]:
fpr, tpr, thr = roc_curve(labels, scores)
roc_auc = auc(fpr, tpr)

fig, ax = plt.subplots(figsize=(6, 5))
ax.plot(tpr, 1/(fpr+1e-9), label=f'HZa tagger  AUC={roc_auc:.4f}')
ax.set_xlabel('Signal efficiency')
ax.set_ylabel('1 / Background efficiency')
ax.set_yscale('log')
ax.set_xlim(0, 1)
ax.legend()
ax.set_title('ROC — a-jet vs other')
plt.tight_layout()

In [ ]:
# Score distributions
bins = np.linspace(0, 1, 50)
fig, ax = plt.subplots(figsize=(6, 4))
ax.hist(scores[labels==1], bins=bins, density=True, alpha=0.6, label='a-jet (signal)')
ax.hist(scores[labels==0], bins=bins, density=True, alpha=0.6, label='other (background)')
ax.set_xlabel('P(a-jet)')
ax.set_ylabel('Normalised counts')
ax.legend()
plt.tight_layout()

In [ ]:
# Working points
for wp in [0.85, 0.70, 0.50]:
    idx = np.searchsorted(tpr, wp)
    if idx < len(tpr):
        print(f'sig_eff={wp:.0%}  score>{thr[idx]:.4f}  bkg_eff={fpr[idx]:.4f}  (1/bkg={1/max(fpr[idx],1e-9):.1f})')